In [1]:
import tensorflow as tf
import numpy as np
import random
from sklearn.metrics import accuracy_score
from tensorflow.keras.models import Model
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense

# ================= Load CIFAR-100 dataset =================
(trainX, trainY), (testX, testY) = tf.keras.datasets.cifar100.load_data(label_mode='fine')
trainY, testY = trainY.flatten(), testY.flatten()

# ================= Randomly select 20 classes =================
selected_classes = random.sample(range(100), 20)
class_map = {original: new for new, original in enumerate(selected_classes)}

test_mask = np.isin(testY, selected_classes)
testX, testY = testX[test_mask], testY[test_mask]
testY = np.vectorize(class_map.get)(testY)

# ================= Model configurations =================
models_config = [
    {"name": "Xception", "class": tf.keras.applications.Xception, "preprocess": tf.keras.applications.xception.preprocess_input, "input_shape": (299, 299, 3)},
    {"name": "VGG16", "class": tf.keras.applications.VGG16, "preprocess": tf.keras.applications.vgg16.preprocess_input, "input_shape": (224, 224, 3)},
    {"name": "VGG19", "class": tf.keras.applications.VGG19, "preprocess": tf.keras.applications.vgg19.preprocess_input, "input_shape": (224, 224, 3)},
    {"name": "ResNet50", "class": tf.keras.applications.ResNet50, "preprocess": tf.keras.applications.resnet50.preprocess_input, "input_shape": (224, 224, 3)},
    {"name": "InceptionV3", "class": tf.keras.applications.InceptionV3, "preprocess": tf.keras.applications.inception_v3.preprocess_input, "input_shape": (299, 299, 3)},
    {"name": "MobileNet", "class": tf.keras.applications.MobileNet, "preprocess": tf.keras.applications.mobilenet.preprocess_input, "input_shape": (224, 224, 3)},
    {"name": "MobileNet", "class": tf.keras.applications.MobileNet, "preprocess": tf.keras.applications.mobilenet.preprocess_input, "input_shape": (224, 224, 3)},
    {"name": "EfficientNetB1", "class": tf.keras.applications.EfficientNetB1, "preprocess": tf.keras.applications.efficientnet.preprocess_input, "input_shape": (240, 240, 3)},
    {"name": "NASNetMobile", "class": tf.keras.applications.NASNetMobile, "preprocess": tf.keras.applications.nasnet.preprocess_input, "input_shape": (224, 224, 3)},
    {"name": "NASNetLarge", "class": tf.keras.applications.NASNetLarge, "preprocess": tf.keras.applications.nasnet.preprocess_input, "input_shape": (331, 331, 3)},
]

# ================= Run each model =================
for cfg in models_config:
    print(f"\n========== Running {cfg['name']} ==========")

    # Resize and preprocess test data
    testX_resized = tf.image.resize(testX, cfg["input_shape"][:2]).numpy()
    testX_preprocessed = cfg["preprocess"](testX_resized)

    # Load pretrained model
    base_model = cfg["class"](include_top=False, weights="imagenet", input_shape=cfg["input_shape"])
    base_model.trainable = False

    x = base_model.output
    x = GlobalAveragePooling2D()(x)
    output = Dense(20, activation="softmax")(x)
    model = Model(inputs=base_model.input, outputs=output, name=f"{cfg['name']}-model")

    # Show summary
    model.summary(show_trainable=False)

    # Predict and evaluate
    y_pred_probs = model.predict(testX_preprocessed)
    predY = np.argmax(y_pred_probs, axis=1)
    acc = accuracy_score(testY, predY) * 100
    print(f"Test accuracy on selected 20 classes: {acc:.2f}%")


Output hidden; open in https://colab.research.google.com to view.